# Qwen3-0.6B DoRA+ — 10k Fine-Tuning

This notebook runs one v2.0 experiment: **DoRA+**, **Qwen3-0.6B**, and configuration **D** (full attention + MLP, rank 32).

The runtime-only configuration below is based on config D, with batch size **16**, gradient accumulation **1**, effective batch **16**, learning rate **5e-5**, and **4 epochs**. It does not modify repository files.

## 1. Clone the repository and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull

%pip install -q torch torchao transformers accelerate "peft>=0.14.0" trl datasets bitsandbytes huggingface_hub python-dotenv sentencepiece protobuf
print(f"Repository: {REPO_DIR}")

## 2. Load the Hugging Face token and verify the GPU

Add `HF_TOKEN` to Colab Secrets. It is required to upload the trained adapter and reports.

In [ ]:
import os
import torch
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets before training.")
os.environ["HF_TOKEN"] = hf_token

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this run.")

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB VRAM)")

## 3. Set the 10k runtime configuration

This changes only the active Colab kernel's in-memory copy of config D. The repository's source files remain untouched.

In [ ]:
import runpy
import sys
from pathlib import Path

SRC_DIR = Path(REPO_DIR) / "finetune_DoRAplus" / "src"
DATA_DIR = Path(REPO_DIR) / "finetune_DoRAplus" / "data"
MODEL = "qwen3-0.6b"
CONFIG = "D"
DATASET_SIZE = "10k"
DEVICE = "cuda"
PER_DEVICE_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from finetune_lib import LORA_CONFIGS, LORAPLUS_CONFIGS

runtime_config = {
    **LORA_CONFIGS[CONFIG],
    "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
}
LORA_CONFIGS[CONFIG] = runtime_config
LORAPLUS_CONFIGS[CONFIG] = {**LORAPLUS_CONFIGS[CONFIG], **runtime_config}

print(f"Model: {MODEL}")
print(f"Config: {CONFIG} (same adapter scope as repository config D)")
print(f"Per-device batch: {PER_DEVICE_BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch: {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate: {runtime_config['learning_rate']:.1e}")
print(f"Epochs: {runtime_config['num_train_epochs']}")

def run_repo_entrypoint(script_name: str, arguments: list[str]) -> None:
    previous_argv = sys.argv
    try:
        sys.argv = [str(SRC_DIR / script_name), *arguments]
        runpy.run_path(str(SRC_DIR / script_name), run_name="__main__")
    finally:
        sys.argv = previous_argv

## 4. Prepare the 10k 80/10/10 data split

In [ ]:
import subprocess

prepare_cmd = [
    sys.executable, "-u", str(SRC_DIR / "prepare_doraplus_data.py"),
    "--dataset-size", DATASET_SIZE,
    "--out-dir", str(DATA_DIR),
]
subprocess.run(prepare_cmd, cwd=REPO_DIR, check=True)

## 5. Optional 10-step smoke test

Run this on a fresh Colab runtime before full training. It uses the same runtime-only configuration and does not upload an adapter.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    run_repo_entrypoint(
        "doraplus_train.py",
        [
            "--model", MODEL,
            "--lora-config", CONFIG,
            "--dataset-size", DATASET_SIZE,
            "--device", DEVICE,
            "--gradient-checkpointing",
            "--smoke-test",
            "--no-push",
        ],
    )

## 6. Train the adapter

This runs the full 10k experiment and uploads the adapter and report to Hugging Face.

In [ ]:
run_repo_entrypoint(
    "doraplus_train.py",
    [
        "--model", MODEL,
        "--lora-config", CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--device", DEVICE,
        "--gradient-checkpointing",
    ],
)

## 7. Validate the locally saved adapter

In [ ]:
run_repo_entrypoint(
    "doraplus_validate.py",
    [
        "--model", MODEL,
        "--lora-config", CONFIG,
        "--dataset-size", DATASET_SIZE,
        "--split", "val",
        "--device", DEVICE,
        "--local",
    ],
)

## 8. Optional locked test evaluation

Set the switch to `True` only after reviewing validation results.

In [ ]:
RUN_TEST_EVALUATION = False

if RUN_TEST_EVALUATION:
    run_repo_entrypoint(
        "doraplus_validate.py",
        [
            "--model", MODEL,
            "--lora-config", CONFIG,
            "--dataset-size", DATASET_SIZE,
            "--split", "test",
            "--device", DEVICE,
            "--local",
        ],
    )